## Challenge A - cyber

This section checks that the distributed development data match the handout before any model is fitted. It loads three files:
- `cyber_development.csv`: the model table (one row per network flow)
- `cyber_groups.csv`: the grouping key for validation
- `cyber_class_codes.json`: the mapping from target code to class name

It then checks the structure, integrity, class supports and the meaning of `feature_group`. Nothing is fitted here and no row is removed or modified

In [1]:
# --- Challenge A: setup ---
# Imports library versions, project seed and data path
# This block is self-contained: it does not use anything from Challenge B
import sys
import json
import pandas as pd
import numpy as np
import sklearn

# Fixed seed for every random step
# Given by the project guide
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| sklearn", sklearn.__version__)
# Data paths, relative to the notebook location
DATA_DIR = "../../datos"
CYBER_DIR = DATA_DIR + "/cyber"

PROJECT_RANDOM_STATE = 2627
python 3.14.6 | pandas 3.0.3 | sklearn 1.9.0


In [2]:
# --- Challenge A: load development table, groups and class codes ---
# Main table: one row per network connection, with its features and class
cyber = pd.read_csv(CYBER_DIR + "/cyber_development.csv")

# Groups file: tells which group each row belongs to (for validation)
cyber_groups = pd.read_csv(CYBER_DIR + "/cyber_groups.csv")

# Class codes: dictionary that maps each code to its name
with open(CYBER_DIR + "/cyber_class_codes.json") as f:
    cyber_codes = json.load(f)

# shape = (rows, columns). Both files should have the same number of rows
print("cyber:", cyber.shape, "| groups:", cyber_groups.shape)
print("class codes:", cyber_codes)

# Data type of each column
print(cyber.dtypes)

cyber: (34928, 13) | groups: (34928, 2)
class codes: {'0': 'backdoor', '1': 'ddos', '2': 'dos', '3': 'injection', '4': 'mitm', '5': 'normal', '6': 'password', '7': 'ransomware', '8': 'scanning', '9': 'xss'}
row_id              str
proto               str
service             str
conn_state          str
duration        float64
src_bytes         int64
dst_bytes         int64
missed_bytes      int64
src_pkts          int64
src_ip_bytes      int64
dst_pkts          int64
dst_ip_bytes      int64
attack_code       int64
dtype: object


In [3]:
# --- Challenge A: integrity checks ---
# Goal: make sure the data is clean before using it

# 1. Inspect the groups file: column names and first rows
print("groups columns:", cyber_groups.columns.tolist())
print(cyber_groups.head(3))

# 2. Check that both files list the rows in the same order
# Comparing row_id element by element gives an array of True/False
# .all() returns True only if EVERY comparison is True
# This matters because we will later attach each row to its group position
print("row_id aligned with groups file:", (cyber["row_id"].values == cyber_groups["row_id"].values).all())

# 3. Count duplicated row_id values. Every row must have a unique identifier
print("duplicated row_id:", cyber["row_id"].duplicated().sum())

# 4. Count missing values in each column
print("missing values per column:")
print(cyber.isna().sum())

# 5. Numeric predictors must be >= 0 so that the course-fixed pipeline is defined
num_cols = cyber.select_dtypes("number").columns.drop("attack_code")
print("numeric predictors:", list(num_cols))
print("numeric predictors >= 0:", (cyber[num_cols] >= 0).all().all())
# Range of each numeric predictor: smallest non-zero value and maximum
print(pd.DataFrame({"min_nonzero": cyber[num_cols][cyber[num_cols]>0].min(), "max": cyber[num_cols].max()}))

# 6. Schema against the handout: row_id + 11 predictors + integer target with the JSON codes
predictors = [c for c in cyber.columns if c not in ("row_id", "attack_code")]
print("number of predictors:", len(predictors), "(handout: 11)")
print("target is integer:", pd.api.types.is_integer_dtype(cyber["attack_code"]))
print("target codes == JSON codes:", set(cyber["attack_code"].unique()) == {int(k) for k in cyber_codes})

groups columns: ['row_id', 'feature_group']
   row_id  feature_group
0  C00000              0
1  C00001              1
2  C00002              2
row_id aligned with groups file: True
duplicated row_id: 0
missing values per column:
row_id          0
proto           0
service         0
conn_state      0
duration        0
src_bytes       0
dst_bytes       0
missed_bytes    0
src_pkts        0
src_ip_bytes    0
dst_pkts        0
dst_ip_bytes    0
attack_code     0
dtype: int64
numeric predictors: ['duration', 'src_bytes', 'dst_bytes', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes']
numeric predictors >= 0: True
              min_nonzero           max
duration         0.000001  9.337206e+04
src_bytes        1.000000  1.430386e+09
dst_bytes        1.000000  1.253827e+09
missed_bytes     1.000000  1.854527e+09
src_pkts         1.000000  2.462300e+04
src_ip_bytes    28.000000  6.522626e+06
dst_pkts         1.000000  6.569000e+04
dst_ip_bytes    28.000000  8.639552e+07
nu

In [4]:
# --- Challenge A: class distribution and groups ---
# 1. Number of rows per class
# value_counts() counts how many times each code appears
# sort_index() orders them by code instead of by frequency
counts = cyber["attack_code"].value_counts().sort_index()

# Table with three columns:
#    name: class name
#    n: number of rows in that class
#    pct: percentage of the total, rounded to 2 decimals
class_dist = pd.DataFrame({
    "name": [cyber_codes[str(k)] for k in counts.index],
    "n": counts.values,
    "pct": (100 * counts / len(cyber)).round(2).values,
}, index=counts.index)
print(class_dist)

# 2. Number of groups and number of rows per group
# nunique() counts distinct values
# describe() summarises group sizes: mean, min, max, etc.
cyber_g = cyber_groups.iloc[:,1]
print("\nnumber of groups:", cyber_g.nunique())
print("rows per group:")
print(cyber_g.value_counts().describe())

# 3. Categories in the text columns and how often each one appears
# Useful to spot very rare categories, which can cause problems during validation
for col in ["proto", "service", "conn_state"]:
    print(f"\n{col}:", cyber[col].value_counts().to_dict())

                   name     n    pct
attack_code                         
0              backdoor  1573   4.50
1                  ddos  4800  13.74
2                   dos  4800  13.74
3             injection  4800  13.74
4                  mitm   805   2.30
5                normal  4800  13.74
6              password  4800  13.74
7            ransomware   208   0.60
8              scanning  3542  10.14
9                   xss  4800  13.74

number of groups: 34883
rows per group:
count    34883.000000
mean         1.001290
std          0.041798
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          4.000000
Name: count, dtype: float64

proto: {'tcp': 28423, 'udp': 6431, 'icmp': 74}

service: {'-': 16454, 'http': 11220, 'dns': 6296, 'ssl': 508, 'ftp': 346, 'smb': 81, 'smb;gssapi': 15, 'dce_rpc': 7, 'gssapi': 1}

conn_state: {'SF': 14640, 'REJ': 7497, 'S0': 4102, 'S3': 2979, 'S1': 1899, 'RSTR': 1048, 'SHR': 1034, 'RSTO': 633, 'OTH': 400, 'SH'

In [5]:
# --- Challenge A: rows sharing a feature_group
# Are rows that share a group exact copies? Do they have the same class?

# 1. Add the group column to the data. Doing it by position is safe because the integrity check cell above confirmed that the row order matches
cyber_with_group = cyber.assign(feature_group=cyber_groups["feature_group"].values)

# 2. For each row, compute how many rows its group has
# transform("size") returns one value per row, not one per group
group_size = cyber_with_group.groupby("feature_group")["row_id"].transform("size")

# 3. Keep only rows that belong to groups with 2 or more rows
dup = cyber_with_group[group_size > 1]
print("rows in multi-row groups:", len(dup), "| groups:", dup["feature_group"].nunique())

# 4. Model input columns: everything except the identifier and the class
feat_cols = [c for c in cyber.columns if c not in ("row_id", "attack_code")]

# 5. Are they exact copies? For each group, count the distinct values in each column. If every column has 1, all rows in the group are identical
n_values = dup.groupby("feature_group")[feat_cols].nunique()
identical = (n_values == 1).all(axis=1) # True if EVERY column has 1 value
print("groups with identical features:", identical.sum())

# 6. Same class? Count the distinct classes in each group. More than one means identical rows with different labels
n_classes = dup.groupby("feature_group")["attack_code"].nunique()
print("groups with more than one class:", (n_classes > 1).sum())

# 7. If any group has several classes, show those rows with the class name
conflicts = n_classes[n_classes > 1].index
conflict_rows = dup[dup["feature_group"].isin(conflicts)]
print(conflict_rows[["feature_group", "attack_code"]]
      .assign(name=conflict_rows["attack_code"].astype(str).map(cyber_codes))
      .sort_values("feature_group"))

# 8. Minimum number of unavoidable errors: in each group, the best any model can do is predict the most
# frequent class, so it gets at most that many right
max_correct = (dup.groupby("feature_group")["attack_code"].agg(lambda s: s.value_counts().max()).sum())
print("unavoidable errors (minimum):", len(dup) - max_correct)

# 9. Reverse check: identical feature vectors never sit in different groups
print("distinct feature vectors == groups:", len(cyber[feat_cols].drop_duplicates()) == cyber_with_group["feature_group"].nunique())

rows in multi-row groups: 83 | groups: 38
groups with identical features: 38
groups with more than one class: 38
       feature_group  attack_code        name
712              900            7  ransomware
11515            900            1        ddos
23261            900            4        mitm
1162            1461            5      normal
12949           1461            7  ransomware
...              ...          ...         ...
29524          30307            7  ransomware
33322          33667            8    scanning
26958          33667            2         dos
28257          35301            3   injection
31209          35301            1        ddos

[83 rows x 3 columns]
unavoidable errors (minimum): 45
distinct feature vectors == groups: True


## Challenge A - findings

**Schema:** 34.928 rows x 13 columns: `row_id`(identifier only, never a predictor), 3 nominal and 8 numeric predictors, and the target `attack_code` (10 codes, all present and named in the JSON)

**Integrity:** groups file aligned row by row; no duplicated `row_id`, no missing values; all numeric predictors >= 0, so `log1p` is defined 

**Scales:** the byte counts go from a smallest non-zero value of 1 up to about 1.9x10⁹ (about nine orders of magnitude), and `duration` goes from 10⁻⁶ to 9.3x10⁴. On the raw scale a Euclidean distance would be dominated by the byte counts 

**Imbalance:** ransomware has only 208 rows, about 42 per fold with 5 folds, so one row changes its fold recall by ~2.4 points. Accuracy alone is not enough: report per-class with its spread across folds

**Rare categories:** `service=gssapi` has a single row, `dce_rpc` 7 and `smb;gssapi` 15; `conn_state = RSTOS0` has 32 and `proto = icmp` 74. A category with one row is necessarily absent from the training portion of the fold where it is validated, so the encoder must be fitted inside each fold with `handle_unknown="ignore"`

**Feature groups:** 38 of the 34.883 groups have more than one row (83 rows, at most 4 per group). Rows in the same group are exact copies, and identical feature vectors never sit in different groups, so each `feature_group`is exactly one distinct feature vector. Grouped validation keeps every copy on the same side.

**Label conflicts:** all 38 groups mix classes, ransomware included; in fact every copy inside a group carries a different label. At least 45 errors are unavoidable: maximum accuracy (34.928 - 45) / 34.928 = 99.87 %. The rows are kept: the modelling table is fixed.

## Challenge B - storage

This section checks that the distributed development data match the handout before any model is fitted. It loads `storage_development.csv` (one row per storage test: its configuration and the measured latency) and `storage_groups.csv` (the configuration each row belongs to). It then checks the structure, integrity, the meaning of `config_group`, the value grid of each predictor and the distribution of the target. Nothing is fitted here and no row is removed or modified

In [6]:
# --- Challenge B: setup ---
# This block is self-contained: it does not use anything from Challenge A
import sys
import json
import pandas as pd
import numpy as np
import sklearn

# Fixed seed for every random step
# Given by the project guide
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| sklearn", sklearn.__version__)
# Data paths, relative to the notebook location
DATA_DIR = "../../datos"
STORAGE_DIR = DATA_DIR + "/storage"

PROJECT_RANDOM_STATE = 2627
python 3.14.6 | pandas 3.0.3 | sklearn 1.9.0


In [7]:
# --- Challenge B: load development table and groups ---
# Main table: one row per storage test, with its configuration and latency
storage = pd.read_csv(STORAGE_DIR + "/storage_development.csv")

# Groups file: tells which configuration each row belongs to
storage_groups = pd.read_csv(STORAGE_DIR + "/storage_groups.csv")

# shape = (rows, columns). Both files should have the same number of rows
print("storage:", storage.shape, "| groups:", storage_groups.shape)

# Data type of each column
print(storage.dtypes)

storage: (12190, 11) | groups: (12190, 2)
row_id               str
block_size         int64
n_jobs             int64
iodepth            int64
read_fraction      int64
load_type            str
io_type              str
raid                 str
n_disks            int64
device_type          str
latency_ns       float64
dtype: object


In [8]:
# --- Challenge B: integrity checks, groups and value ranges ---

# 1. Inspect the groups file
print("groups columns:", storage_groups.columns.tolist())
print(storage_groups.head(3))

# 2. Check that both files list the rows in the same order
print("row_id aligned with groups file: ", (storage["row_id"].values == storage_groups["row_id"].values).all())

# 3. Duplicated row_id values and missing values
print("duplicated row_id:", storage["row_id"].duplicated().sum())
print("missing values per column:")
print(storage.isna().sum())

# 4. Number of groups and number of rows per group
storage_g = storage_groups.iloc[:, 1]
print("\nnumber of groups:", storage_g.nunique())
print("rows per group:")
print(storage_g.value_counts().describe())

# 5. Sorted distinct values of each input variable
for col in ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks", "load_type", "io_type", "raid", "device_type"]:
    print(f"\n{col}:", sorted(storage[col].unique().tolist()))

# 6. Summary of the target
print("\nlatency_ns:")
print(storage["latency_ns"].describe())

# 7. Check that every latency is greater than 0
print("min > 0:", (storage["latency_ns"] > 0).all())

# 8. Each config_group is one configuration: its 5 replicates share all 9 features
feat_cols = [c for c in storage.columns if c not in ("row_id", "latency_ns")]
print("number of predictors:", len(feat_cols), "(handout: 9)")
print("iops absent:", "iops" not in storage.columns)
print("target finite:", np.isfinite(storage["latency_ns"]).all())
storage_with_group = storage.assign(config_group=storage_g.values)
print("replicates identical within group:", (storage_with_group.groupby("config_group")[feat_cols].nunique() == 1).all().all())
print("distinct configurations == groups:", len(storage[feat_cols].drop_duplicates()) == storage_g.nunique())

groups columns: ['row_id', 'config_group']
   row_id  config_group
0  S00000             0
1  S00001             0
2  S00002             0
row_id aligned with groups file:  True
duplicated row_id: 0
missing values per column:
row_id           0
block_size       0
n_jobs           0
iodepth          0
read_fraction    0
load_type        0
io_type          0
raid             0
n_disks          0
device_type      0
latency_ns       0
dtype: int64

number of groups: 2438
rows per group:
count    2438.0
mean        5.0
std         0.0
min         5.0
25%         5.0
50%         5.0
75%         5.0
max         5.0
Name: count, dtype: float64

block_size: [4, 8, 16, 32, 64, 128, 256, 512, 1024]

n_jobs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64]

iodepth: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]

rea

## Challenge B - findings

**Schema:** 12.190 rows x 11 columns: `row_id`, 5 integer and 4 nominal predictors, and the target `latency_ns`. `iops` is absent, as rule 4 requires.

**Integrity:** groups file aligned row by row; no duplicated `row_id`, no missing values; the target is finite and strictly positive.

**Configuration groups:** 2.438 groups of exactly 5 rows. The 5 replicates of a group share all 9 predictors, and the number of distinct configurations equals the number of groups, so no configuration is split across groups: each `config_group` is exactly one configuration. The unit of generalisation is the configuration, so validation must keep the 5 replicates together.

**Scales:** predictors lie on discrete grids with very different ranges (`iodepth` 1-32, `block_size` 4-1024), so distance-based methods need standardisation inside each fold.

**Target:** from 1.16x10⁵ to 2.53x10⁹ ns (~4.3 orders of magnitude); the mean is 23 times the median, so the distribution has a long right tail. The target representation is justified in P2.3.